# 08 空間流病 — 練習

用松柏護理之家退伍軍人症資料練習空間分析與視覺化。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## 題目 1：致死率空間分布

1. 計算每個 floor × wing 的致死率（CFR = 死亡 / 感染 × 100）
2. 用 `sns.heatmap()` 畫致死率熱力圖
3. 哪個翼區致死率最高？致死率高的翼區是否也是侵襲率最高的翼區？

In [ ]:
# TODO: 計算 floor × wing 致死率
# TODO: sns.heatmap()
# TODO: 解讀

## 題目 2：淋浴使用的空間分布

我們在 Ch05 發現淋浴使用（`shower_use`）是感染的危險因子。

1. 計算每個 floor × wing 中淋浴使用者的比例
2. 用熱力圖呈現
3. 淋浴比例高的翼區是否也是侵襲率高的翼區？
4. 這個觀察結果支持「水源系統是傳播途徑」的假說嗎？

In [ ]:
# TODO: 計算 floor × wing 淋浴使用比例
# TODO: 熱力圖
# TODO: 與侵襲率熱力圖並排比較

## 題目 3（挑戰題）：高風險房間清單

你要向感控團隊提交一份「高風險房間清單」：

1. 計算每間房的侵襲率
2. 篩選侵襲率 ≥ 75% 的房間
3. 產出一個表格，包含：`room`, `total`, `infected`, `attack_rate`, `floor`, `wing`
4. 按侵襲率降序排列
5. 高風險房間是否集中在特定翼區？

In [ ]:
# TODO: 計算每間房侵襲率
# TODO: 篩選 >= 75%
# TODO: 整理表格並排序
# TODO: 哪些翼區最多高風險房間？

## 題目 4：登革熱各行政區發生率（登革熱情境）

某縣市登革熱流行，衛生局取得各行政區病例與人口。

1. 計算各行政區病例數與每十萬人發生率
2. 找出發生率最高的行政區（未必是病例數最多者）
3. 畫出各區發生率長條圖
4. 解讀：為何要用發生率而非絕對病例數比較各區風險？

In [ ]:
# 登革熱：各行政區病例（積水多的安南區風險偏高）
rng = np.random.default_rng(841)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
rate_per_100k = {"安南區": 22, "三民區": 8, "北屯區": 5, "板橋區": 4, "中西區": 9}
_recs = []
for d in districts:
    n = rng.poisson(rate_per_100k[d] * pop[d] / 100000)
    for _ in range(n):
        _recs.append({"district": d, "age": int(rng.integers(5, 85)),
                      "serotype": rng.choice(["DENV-1", "DENV-2", "DENV-3"])})
dengue = pd.DataFrame(_recs)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"登革熱通報 {len(dengue)} 例，橫跨 {dengue['district'].nunique()} 個行政區")

# TODO: 用 groupby 算各行政區的登革熱病例數
# TODO: 合併 region_pop，計算每十萬人發生率 = 病例數 / 人口 * 100000
# TODO: 找出「發生率」最高的行政區（提示：不一定是病例數最多的那一區）
# TODO: 用長條圖（df.plot.bar 或 sns.barplot）畫出各區發生率並排序
# TODO: 解讀——為什麼要用發生率而非絕對病例數比較各區風險？

## 題目 5：COVID-19 區域侵襲率熱區圖（COVID-19 情境）

將城市切成 4×5 網格，取得各區域人口與病例。

1. 計算每個區域的侵襲率（%）
2. 用 `pivot` 排成 row×col 網格，`sns.heatmap` 畫熱區圖
3. 找出侵襲率最高的區域，說明熱區集中在哪一側

In [ ]:
# COVID-19：4x5 網格區域的人口與病例（北部 A/B 列風險較高）
rng = np.random.default_rng(852)
_recs = []
for r in list("ABCD"):
    for c in range(1, 6):
        popn = int(rng.integers(2000, 6000))
        base = 0.03 + (0.05 if r in ("A", "B") else 0.0) + rng.normal(0, 0.008)
        cases = rng.binomial(popn, max(0.005, base))
        _recs.append({"region": f"{r}{c}", "row": r, "col": c, "population": popn, "cases": cases})
covid = pd.DataFrame(_recs)
print(f"COVID-19：{len(covid)} 個區域，總人口 {covid['population'].sum():,}，總病例 {covid['cases'].sum()}")

# TODO: 計算每個區域的侵襲率（%）= cases / population * 100
# TODO: 用 pivot（index=row, columns=col, values=侵襲率）排成網格
# TODO: 用 sns.heatmap（annot=True）畫出侵襲率熱區圖
# TODO: 找出侵襲率最高的區域，並說明熱區集中在哪一側

## 題目 6：腸病毒班級侵襲率熱區圖（腸病毒情境）

某國小腸病毒群聚，取得各年級各班學生數與病例。

1. 計算年級×班級的侵襲率
2. 用 `pivot` + `sns.heatmap` 畫年級×班級矩陣
3. 比較各年級整體侵襲率，解讀低／高年級差異

In [ ]:
# 腸病毒：國小各年級各班的學生數與病例（低年級風險較高）
rng = np.random.default_rng(863)
_recs = []
for g in range(1, 7):
    for cl in range(1, 6):
        students = int(rng.integers(25, 35))
        risk = max(0.02, 0.28 - g * 0.03)
        cases = rng.binomial(students, risk)
        _recs.append({"grade": g, "classroom": cl, "students": students, "cases": cases})
ev = pd.DataFrame(_recs)
print(f"腸病毒：{ev['grade'].nunique()} 個年級 × {ev['classroom'].nunique()} 班，共 {ev['cases'].sum()} 例")

# TODO: 計算每個年級 × 班級的侵襲率（%）= cases / students * 100
# TODO: 用 pivot（index=grade, columns=classroom）排成年級 × 班級矩陣
# TODO: 用 sns.heatmap 畫出侵襲率，觀察哪個年級整體偏高
# TODO: 解讀——低年級與高年級的侵襲率差異可能反映什麼？

## 題目 7：諾羅病毒宴會 spot map（諾羅病毒情境）

一場宴會後爆發諾羅病毒，取得各桌座位座標、出席人數與病例。

1. 計算每桌侵襲率
2. 用 `scatter` 畫 spot map（點大小=人數、顏色=侵襲率）
3. 標出高侵襲率的群聚桌次，推論可能汙染源位置

In [ ]:
# 諾羅病毒：宴會 25 桌的座位平面圖（靠近海鮮區的桌次侵襲率高）
rng = np.random.default_rng(874)
_recs = []
for t in range(1, 26):
    x, y = (t - 1) % 5, (t - 1) // 5
    attendees = int(rng.integers(8, 12))
    near_seafood = (x <= 1 and y <= 1)   # 左下角靠海鮮區
    ar = 0.6 if near_seafood else 0.1
    cases = rng.binomial(attendees, ar)
    _recs.append({"table": t, "x": x, "y": y, "attendees": attendees, "cases": cases})
noro = pd.DataFrame(_recs)
print(f"諾羅病毒宴會：{len(noro)} 桌，{noro['attendees'].sum()} 人出席，{noro['cases'].sum()} 人發病")

# TODO: 計算每桌的侵襲率 = cases / attendees
# TODO: 用 scatter 畫散點圖模擬宴會平面圖（x, y 為座位、點大小=出席人數、顏色=侵襲率）
# TODO: 標出侵襲率明顯偏高的「群聚」桌次
# TODO: 解讀——這個空間群聚指向哪個可能的汙染源？

## 題目 8（挑戰題）：結核病鄉鎮空間分析（結核情境）

12 個鄉鎮的結核病通報，人口規模差異很大。

1. 計算各鄉鎮每十萬人發生率
2. 比較「病例數」與「發生率」排序前段名單是否相同
3. 指出小人口鄉鎮發生率不穩定的原因（小分母問題）
4. 計算擁擠指數與發生率的相關
5. 解讀：畫鄉鎮地圖該呈現病例數還是發生率？如何處理小人口區？

In [ ]:
# 結核病：12 個鄉鎮的人口、擁擠指數與病例（人口差異大 → 小分母率不穩）
rng = np.random.default_rng(885)
_recs = []
for i in range(1, 13):
    popn = int(rng.integers(3000, 120000))
    crowding = round(float(rng.uniform(0.5, 2.0)), 2)
    cases = rng.poisson(15 * crowding * popn / 100000)
    _recs.append({"township": f"T{i:02d}", "population": popn,
                  "crowding_index": crowding, "cases": cases})
tb = pd.DataFrame(_recs)
print(f"結核病：{len(tb)} 個鄉鎮，人口 {tb['population'].min():,}–{tb['population'].max():,}")

# TODO: 計算各鄉鎮每十萬人發生率 = cases / population * 100000
# TODO: 分別依「病例數」與「發生率」排序，比較兩份前段名單是否相同
# TODO: 指出人口很小的鄉鎮，其發生率為何可能不穩定（小分母問題）
# TODO: 計算 crowding_index 與發生率的相關，判斷擁擠是否為空間風險因子
# TODO: 解讀——製作結核病鄉鎮地圖時，你會呈現病例數還是發生率？為什麼？

## 🌐 空間統計練習（Moran's I / LISA / Gi\*）

以下題目搭配課文 **Part 3** 與 `08_spatial_statistics.ipynb`，使用真實台灣縣市地圖 + 合成登革熱資料。

In [ ]:
# === 空間統計練習（搭配課文 Part 3 / 08_spatial_statistics.ipynb；需要 esda + libpysal）===
import geopandas as gpd
from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
from esda.smoothing import Empirical_Bayes

gdf = gpd.read_file("data/geojson/county_smooth_inset.geojson")[["COUNTYNAME", "is_inset", "geometry"]]
_pop = {"臺北市":2500000,"新北市":4000000,"桃園市":2270000,"臺中市":2820000,"臺南市":1870000,
        "高雄市":2750000,"基隆市":365000,"新竹市":450000,"嘉義市":265000,"新竹縣":570000,
        "苗栗縣":540000,"彰化縣":1250000,"南投縣":480000,"雲林縣":670000,"嘉義縣":500000,
        "屏東縣":810000,"宜蘭縣":454000,"花蓮縣":320000,"臺東縣":215000,
        "澎湖縣":105000,"金門縣":140000,"連江縣":13000}
_cases = {"臺南市":2100,"高雄市":2600,"屏東縣":710,"嘉義縣":290,"嘉義市":140,"雲林縣":270,
          "彰化縣":275,"臺中市":505,"南投縣":58,"苗栗縣":54,"桃園市":205,"新竹縣":46,"新竹市":32,
          "臺北市":150,"新北市":240,"基隆市":18,"宜蘭縣":18,"花蓮縣":10,"臺東縣":6,
          "澎湖縣":9,"金門縣":5,"連江縣":7}
gdf["population"] = gdf["COUNTYNAME"].map(_pop)
gdf["cases"] = gdf["COUNTYNAME"].map(_cases)
gdf["rate"] = (gdf["cases"] / gdf["population"] * 100000).round(1)   # 每十萬人發生率
main = gdf[~gdf["is_inset"]].reset_index(drop=True)                  # 本島 19 個相連縣市
print(f"本島 {len(main)} 縣市；登革熱發生率範圍 {main['rate'].min()}–{main['rate'].max()} /十萬")

## 題目 9：自己定義「鄰居」（空間權重）

對本島 19 縣市分別建立 **Queen 接壤**權重與 **KNN(k=4)** 權重。

1. 比較兩者的平均鄰居數（`w.mean_neighbors`）
2. 找出 Queen 下鄰居**最多 / 最少**的縣市（`w.cardinalities`）
3. row-standardize（`w.transform = "r"`）後，說明每個縣市的權重和是多少、代表什麼

In [ ]:
# TODO: 建立 Queen 與 KNN(k=4) 權重並比較
# 提示：wq = Queen.from_dataframe(main, use_index=False)
#       wk = KNN.from_dataframe(main, k=4)
#       wq.mean_neighbors, pd.Series(wq.cardinalities)

## 題目 10：全域 Moran's I，親手做「洗牌」

1. 用 `Moran(rate, w)` 算出 Moran's I 與 `p_sim`
2. **親手**把發生率 `np.random.permutation` 洗牌 200 次，每次重算一個空間相似度統計量
3. 比較「真實值」和「洗牌分布」，說明 p 值到底是怎麼來的

In [ ]:
# TODO: 算 esda Moran's I，再親手洗牌 200 次建立參考分布
# 提示：mi = Moran(main['rate'].values, wq, permutations=999)
#       洗牌：np.random.default_rng(1).permutation(y)

## 題目 11：LISA 四象限（換舞台：腸病毒）

改用一組合成**腸病毒**發生率（中部偏高），跑 `Moran_Local`：

1. 標出顯著的 **HH / LL / HL / LH**（注意 esda `.q` 編碼：1=HH、2=LH、3=LL、4=HL）
2. 畫 LISA 群聚圖
3. 特別找出並解讀有沒有 **HL 火苗 / LH 颱風眼** 這種空間離群值

In [ ]:
# TODO: 建立腸病毒發生率 -> Moran_Local -> 標記四象限 -> 畫圖
# 提示：lisa = Moran_Local(rate, wq, permutations=999, seed=8)
#       象限：{1:'HH',2:'LH',3:'LL',4:'HL'}；只保留 p_sim < 0.05

## 題目 12：熱點分析 Gi\* + 「Gi\* vs LISA 差在哪」

對同一份腸病毒發生率跑 `G_Local`（二元權重、`star=True`）：

1. 列出顯著的**熱點 / 冷點**（小樣本看 `p_sim`，冷熱看 `Zs` 正負）
2. 用**集合比對** Gi\* 熱點清單 vs 題目 11 的 **LISA HH** 清單，解釋兩者為什麼不完全一樣

In [ ]:
# TODO: G_Local 熱點分析，並和題目 11 的 LISA HH 做集合比對
# 提示：wb = Queen.from_dataframe(main, use_index=False); wb.transform='B'
#       gi = G_Local(rate, wb, permutations=999, seed=8, star=True)

## 題目 13（挑戰題）：小人口不穩 + MAUP

(a) 用 `Empirical_Bayes` 平滑全部 22 縣市的登革熱率，比較**連江縣**平滑前後的率與**排名**。

(b) 迷你 MAUP：合成一個 8×8 網格（左上角高），聚合成 4×4、2×2，各尺度重算 Moran's I，觀察它怎麼變（甚至變號）。

In [ ]:
# TODO(a): Empirical_Bayes(gdf['cases'], gdf['population']) 平滑，比較連江排名
# TODO(b): 8x8 網格 -> 聚合 4x4/2x2 -> 各算一次 Moran's I